In [1]:
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("marketplace.csv")

In [3]:
df.head()

,id,title,description,price,category,image,seller,college,is_sold,created_at
0,1,Well Maintained Breadboard Kit,I am selling this Breadboard Kit after complet...,209.0,Stationery and Tools,26641916ffd34dd38aa5c37c90061c58.jpg,Sonali,VIT Pune,False,2026-04-09 02:21:55.468037
1,2,Camel Geometry Box (Excellent Condition),Selling this Camel Geometry Box since I have u...,92.0,Stationery and Tools,3bc8ec589e954d01aec36d95b41f600e.jpg,madhvi,MIT-WPU,False,2026-02-06 17:38:55.477217
2,3,Badminton Racket (Excellent Condition),Used this Badminton Racket for college and it ...,993.0,Sports,e5967491e4c14fc1b9038cf36e426786.jpg,Muktai,VIT Pune,False,2026-07-14 08:18:55.493077
3,4,Classmate Exam Writing Pad,I am selling this Exam Writing Pad after compl...,144.0,Stationery and Tools,e1ee8ad19a454a6eac9bfcc8a799764f.jpg,Mohan,PCCOE,True,2026-06-10 12:40:55.497043
4,5,Taparia Workshop Tool Kit,This Workshop Tool Kit is in good condition an...,974.0,Others,dc7d6b6ca4e4404d878e06e70e7354ed.jpg,Sonali,VIT Pune,False,2026-05-20 09:53:55.515526


In [4]:
df.shape

(500, 10)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           500 non-null    int64  
 1   title        500 non-null    str    
 2   description  500 non-null    str    
 3   price        500 non-null    float64
 4   category     500 non-null    str    
 5   image        500 non-null    str    
 6   seller       500 non-null    str    
 7   college      500 non-null    str    
 8   is_sold      500 non-null    bool   
 9   created_at   500 non-null    str    
dtypes: bool(1), float64(1), int64(1), str(7)
memory usage: 35.8 KB


In [7]:
df.describe()
df.columns

Index(['id', 'title', 'description', 'price', 'category', 'image', 'seller',
       'college', 'is_sold', 'created_at'],
      dtype='str')

In [8]:
df = df[
    [
        "id",
        "title",
        "description",
        "college",
        "category",
        "price",
        "is_sold"
    ]
]

In [9]:
df.isnull().sum()

id             0
title          0
description    0
college        0
category       0
price          0
is_sold        0
dtype: int64

In [10]:
df["title"] = df["title"].fillna("")

In [11]:
df.isnull().sum()

id             0
title          0
description    0
college        0
category       0
price          0
is_sold        0
dtype: int64

In [13]:
df["combined_text"] = (
    df["title"]
    + " "
    + df["category"]
    + " "
    + df["description"]
    + " "
    + df["college"]
)

In [14]:
df[
    [
        "title",
        "combined_text"
    ]
].head()

,title,combined_text
0,Well Maintained Breadboard Kit,Well Maintained Breadboard Kit Stationery and ...
1,Camel Geometry Box (Excellent Condition),Camel Geometry Box (Excellent Condition) Stati...
2,Badminton Racket (Excellent Condition),Badminton Racket (Excellent Condition) Sports ...
3,Classmate Exam Writing Pad,Classmate Exam Writing Pad Stationery and Tool...
4,Taparia Workshop Tool Kit,Taparia Workshop Tool Kit Others This Workshop...


In [15]:
def clean_text(text):
    text = text.lower()

    text = re.sub(r'[^a-zA-Z0-9 ]', ' ', text)

    words = text.split()

    remove_words = {
        "excellent",
        "good",
        "condition",
        "sale",
        "selling",
        "used",
        "new",
        "like",
        "well",
        "ready"
    }

    words = [word for word in words if word not in remove_words]

    text = " ".join(words)

    text = re.sub(r'\s+', ' ', text)

    return text.strip()


def remove_duplicate_words(text):
    words = text.split()
    return " ".join(dict.fromkeys(words))

In [16]:
df["combined_text"] = df["combined_text"].apply(clean_text)
df["combined_text"] = df["combined_text"].apply(remove_duplicate_words)

In [17]:
df[
    [
        "combined_text"
    ]
].head()

,combined_text
0,maintained breadboard kit stationery and tools...
1,camel geometry box stationery and tools this s...
2,badminton racket sports this for college and i...
3,classmate exam writing pad stationery and tool...
4,taparia workshop tool kit others this is in an...


In [18]:
df[df["combined_text"] == ""]

,id,title,description,college,category,price,is_sold,combined_text


In [19]:
df.head()

,id,title,description,college,category,price,is_sold,combined_text
0,1,Well Maintained Breadboard Kit,I am selling this Breadboard Kit after complet...,VIT Pune,Stationery and Tools,209.0,False,maintained breadboard kit stationery and tools...
1,2,Camel Geometry Box (Excellent Condition),Selling this Camel Geometry Box since I have u...,MIT-WPU,Stationery and Tools,92.0,False,camel geometry box stationery and tools this s...
2,3,Badminton Racket (Excellent Condition),Used this Badminton Racket for college and it ...,VIT Pune,Sports,993.0,False,badminton racket sports this for college and i...
3,4,Classmate Exam Writing Pad,I am selling this Exam Writing Pad after compl...,PCCOE,Stationery and Tools,144.0,True,classmate exam writing pad stationery and tool...
4,5,Taparia Workshop Tool Kit,This Workshop Tool Kit is in good condition an...,VIT Pune,Others,974.0,False,taparia workshop tool kit others this is in an...


In [24]:



custom_stop_words = {
    "excellent",
    "good",
    "condition",
    "anymore",
    "sale",
    "selling",
    "used",
    "new",
    "like",
    "well",
    "ready"
}

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=1000,
    ngram_range=(1,2)
)


In [25]:
tfidf_matrix = vectorizer.fit_transform(df["combined_text"])

In [26]:
tfidf_matrix.shape

(500, 1000)

In [27]:
vectorizer.get_feature_names_out()[:50]

array(['15', '15 electronics', '20000mah', '20000mah electronics', '24',
       '24 inch', '3511', '3511 electronics', '3m', '3m longer', '450',
       '450 electronics', '512gb', '512gb electronics', '991es',
       '991es plus', 'abraham', 'abraham silberschatz', 'affordable',
       'affordable option', 'anymore', 'anymore choice',
       'anymore immediately', 'anymore regular', 'anymore suitable',
       'anymore working', 'anymore works', 'arduino', 'arduino uno',
       'artificial', 'artificial intelligence', 'asked', 'asked previous',
       'backpack', 'backpack completing', 'backpack lack',
       'backpack wildcraft', 'backup', 'backup performance', 'badminton',
       'badminton racket', 'bag', 'bag furniture', 'bank',
       'bank 20000mah', 'basket', 'basket hostel', 'basketball',
       'basketball sports', 'bat'], dtype=object)

In [28]:
df["combined_text"].iloc[0]


'maintained breadboard kit stationery and tools i am this after completing my coursework for engineering practicals only works perfectly has been reason graduating semester vit pune'

In [29]:
df["category"].value_counts()

category
Stationery and Tools    120
Electronics              70
Furniture                64
Books                    60
Sports                   46
Hostel Essentials        39
Cycles                   38
Notes                    38
Others                   25
Name: count, dtype: int64

In [30]:
"and" in vectorizer.get_stop_words()

True

In [31]:
print(vectorizer.stop_words)

english


In [32]:
similarity_matrix = cosine_similarity(tfidf_matrix)

In [33]:
similarity_matrix.shape

(500, 500)

In [34]:
similarity_matrix

array([[1.        , 0.10388118, 0.05011671, ..., 0.00257244, 0.09637002,
        0.02844915],
       [0.10388118, 1.        , 0.01625617, ..., 0.02364984, 0.06441194,
        0.01199665],
       [0.05011671, 0.01625617, 1.        , ..., 0.0474817 , 0.0376683 ,
        0.0148882 ],
       ...,
       [0.00257244, 0.02364984, 0.0474817 , ..., 1.        , 0.00194376,
        0.14584686],
       [0.09637002, 0.06441194, 0.0376683 , ..., 0.00194376, 1.        ,
        0.02735676],
       [0.02844915, 0.01199665, 0.0148882 , ..., 0.14584686, 0.02735676,
        1.        ]], shape=(500, 500))

In [35]:
product_index = 67

scores = list(enumerate(similarity_matrix[product_index]))

scores[:10]

[(0, np.float64(0.0029497760884825207)),
 (1, np.float64(0.05516119437962261)),
 (2, np.float64(0.0032675430859879476)),
 (3, np.float64(0.031110422837967902)),
 (4, np.float64(0.012005746830427013)),
 (5, np.float64(0.012856374871930669)),
 (6, np.float64(0.0028722141051242727)),
 (7, np.float64(0.012831352889206346)),
 (8, np.float64(0.02602946893641057)),
 (9, np.float64(0.02884424062404929))]

In [36]:
sorted_scores = sorted(
    scores,
    key=lambda x: x[1],
    reverse=True
)

In [37]:
sorted_scores = sorted_scores[1:21]

In [38]:
for index, score in sorted_scores[:5]:
    print(df.iloc[index]["title"])
    print("Similarity:", round(score,3))
    print("-"*40)

Laundry Basket
Similarity: 0.697
----------------------------------------
Bucket
Similarity: 0.532
----------------------------------------
Laundry Basket
Similarity: 0.479
----------------------------------------
Laundry Basket - Like New
Similarity: 0.464
----------------------------------------
Well Maintained Single Mattress
Similarity: 0.424
----------------------------------------


In [39]:
print(df.iloc[0][["title", "category", "description"]])

print(df["title"].nunique())

print(df["category"].value_counts())

title                             Well Maintained Breadboard Kit
category                                    Stationery and Tools
description    I am selling this Breadboard Kit after complet...
Name: 0, dtype: object
297
category
Stationery and Tools    120
Electronics              70
Furniture                64
Books                    60
Sports                   46
Hostel Essentials        39
Cycles                   38
Notes                    38
Others                   25
Name: count, dtype: int64


In [40]:
print(df.iloc[10]["combined_text"])

casio fx 991es plus electronics i am this after completing my coursework maintained minor cosmetic scratches only can be immediately without any issues reason for shifting to another city coep


In [41]:
import joblib

joblib.dump(df, "dataset.pkl")

joblib.dump(vectorizer, "tfidf.pkl")

joblib.dump(similarity_matrix, "similarity.pkl")

joblib.dump(vectorizer, "../models/vectorizer.pkl")

['../models/vectorizer.pkl']

In [42]:
import os

print(os.listdir())

['.ipynb_checkpoints', 'dataset.pkl', 'desc_generator.ipynb', 'marketplace.csv', 'recommendation_system.ipynb', 'rec_func.ipynb', 'similarity.pkl', 'tfidf.pkl']
